# NYC Shelter Intake Log: Exploratory Data Analysis

This notebook adapts the course EDA workflow to the NYC shelter intake CSV using PySpark: inspect the schema, check data quality, summarize selected fields, and visualize aggregate trends. It reads directly from the provided S3 object; it does not copy the source data.

Run on master node:

`
cd ~/spark
./sbin/start-master.sh --host 172.31.31.200
jps
`

Run on worker nodes:

`
cd ~/spark
./sbin/start-worker.sh spark://172.31.31.200:7077
jps
`

## Before you run this notebook

- Run it in the course Spark environment with read access to `s3a://dsan6000-final/nyc-shelters/intake_log.csv`.
- The codebook includes names, free-text intake notes, and Base64-encoded photos, as well as sensitive housing information. This notebook avoids displaying raw records and excludes direct identifiers, free text, and image data from its working DataFrame.
- Confirm with your instructor that the data is synthetic and approved for your intended use. The `synthetic_data` field alone should not be treated as proof that the data is safe to publish. Do not place the raw data or notebook outputs in a public bucket unless that is explicitly approved.
- Several cells perform full-data scans (for example, `count()` and null counts). These are useful for an initial EDA but can take time on large data.

## 1. Connect to Spark and read the CSV

Spark uses the `s3a://` URI for S3 access. Schema inference is convenient for exploration but requires Spark to inspect the data; for a production pipeline, define and validate an explicit schema.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName("NYC Shelter Intake EDA").getOrCreate()

source_path = "s3a://dsan6000-final/nyc-shelters/intake_log.csv"
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .csv(source_path)
)

print(f"Source: {source_path}")

## 2. Inspect structure without showing records

Review the inferred column names and types. Avoid `df.show()` here: it would print actual intake records, including fields that may be sensitive.

In [ ]:
df.printSchema()
print(f"Number of columns: {len(df.columns)}")
print("Columns:", df.columns)

### Row count

This is a full scan of the CSV. It returns only the total, not the underlying records.

In [ ]:
row_count = df.count()
print(f"Number of intake records: {row_count:,}")

## 3. Create a restricted working DataFrame

Keep the original `df` available for aggregate quality checks, but use `eda_df` for later analysis. The excluded columns include identifiers, names, free text, and encoded images. Do not add these fields to a display or export.

In [ ]:
sensitive_columns = [
    "intake_id",
    "client_display_name",
    "presenting_reason",
    "prior_living_situation_reported",
    "primary_barrier",
    "immediate_needs",
    "potential_referral",
    "client_goal",
    "intake_note",
    "intake_photo_b64",
]
excluded_columns = [name for name in sensitive_columns if name in df.columns]
eda_df = df.drop(*excluded_columns)
print(f"Excluded {len(excluded_columns)} sensitive or identifying columns from eda_df.")

## 4. Check missing and blank values

For string columns, whitespace-only values count as missing too. The result contains column names and aggregate counts only.

In [ ]:
missing_expressions = []
for field in df.schema.fields:
    value = F.col(field.name)
    missing = value.isNull()
    if isinstance(field.dataType, StringType):
        missing = missing | (F.trim(value) == "")
    missing_expressions.append(
        F.sum(F.when(missing, 1).otherwise(0)).alias(field.name)
    )

missing_values = df.agg(*missing_expressions).first().asDict()
missingness = spark.createDataFrame(
    [(column, int(count or 0)) for column, count in missing_values.items()],
    ["column", "missing_count"],
).withColumn("missing_pct", F.round(F.col("missing_count") / F.lit(row_count) * 100, 2))

missingness.orderBy(F.desc("missing_pct"), F.asc("column")).show(
    n=len(df.columns), truncate=False
)

## 5. Check the synthetic-data marker and duplicate IDs

These checks report counts only. A marker value is a data-quality signal, not independent confirmation that the records are synthetic.

In [ ]:
if "synthetic_data" in df.columns:
    df.groupBy("synthetic_data").count().orderBy("synthetic_data").show()

if "intake_id" in df.columns:
    duplicate_id_groups = (
        df.groupBy("intake_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    print(f"Number of intake IDs appearing more than once: {duplicate_id_groups:,}")

## 6. Summarize numeric fields

Approximate quartiles give a robust first look at numeric distributions without collecting individual values. Only the aggregate quantiles are brought back to the notebook driver.

In [ ]:
numeric_candidates = [
    "age",
    "children_count",
    "reported_housing_instability_days",
    "follow_up_days",
    "note_word_count",
]
numeric_columns = [name for name in numeric_candidates if name in eda_df.columns]

if numeric_columns:
    quantiles = eda_df.approxQuantile(numeric_columns, [0.25, 0.5, 0.75], 0.01)
    for name, values in zip(numeric_columns, quantiles):
        print(f"{name} (25th, 50th, 75th percentiles): {values}")
else:
    print("No expected numeric columns were found. Check the inferred schema.")

## 7. Explore selected categorical fields

These are intended to be categorical fields from the codebook. Rare categories are suppressed with a minimum count of five; this is a basic disclosure precaution, not a guarantee of anonymity. Review results before sharing them.

In [ ]:
categorical_columns = [
    "household_type",
    "program_type",
    "contact_type",
    "employment_status_reported",
    "income_status_reported",
    "referral_status",
    "housing_eligibility_verified",
]
categorical_columns = [name for name in categorical_columns if name in eda_df.columns]

for name in categorical_columns:
    print(f"\n{name}: top categories (groups with fewer than 5 records omitted)")
    (
        eda_df.groupBy(name)
        .count()
        .filter(F.col("count") >= 5)
        .orderBy(F.desc("count"))
        .limit(20)
        .show(truncate=True)
    )

## 8. Parse intake dates and check date quality

The conversion below expects Spark-recognized date strings (commonly `yyyy-MM-dd`). First inspect the parse-success counts; if the source uses another format, pass that format as the second argument to `to_date`.

In [ ]:
if "intake_date" in df.columns:
    dated_df = df.withColumn(
        "_intake_date_parsed",
        F.to_date(F.col("intake_date").cast("string")),
    )
    date_quality = dated_df.agg(
        F.count("_intake_date_parsed").alias("parsed_dates"),
        F.sum(F.col("_intake_date_parsed").isNull().cast("long")).alias("missing_or_unparsed"),
        F.min("_intake_date_parsed").alias("earliest_date"),
        F.max("_intake_date_parsed").alias("latest_date"),
    )
    date_quality.show(truncate=False)
else:
    print("intake_date was not found. Check the schema and codebook.")

### Intakes by month

The chart is based on monthly aggregate counts, not individual records.

In [ ]:
if "intake_date" in df.columns:
    monthly_intakes = (
        dated_df.filter(F.col("_intake_date_parsed").isNotNull())
        .groupBy(F.date_format("_intake_date_parsed", "yyyy-MM").alias("month"))
        .count()
        .orderBy("month")
    )
    monthly_intakes.show(n=120, truncate=False)

    monthly_pdf = monthly_intakes.limit(120).toPandas()
    if not monthly_pdf.empty:
        import matplotlib.pyplot as plt

        ax = monthly_pdf.plot(x="month", y="count", kind="line", marker="o", legend=False)
        ax.set(title="Shelter intakes by month", xlabel="Intake month", ylabel="Records")
        ax.tick_params(axis="x", labelrotation=60)
        plt.tight_layout()
        plt.show()

## 9. Follow-up timing

Compare the intake date with the follow-up target date. The output is aggregated; it does not expose the dates or other fields for individual clients.

In [ ]:
if "intake_date" in df.columns and "follow_up_target_date" in df.columns:
    follow_up_df = (
        df.withColumn("_intake_date", F.to_date(F.col("intake_date").cast("string")))
        .withColumn(
            "_target_date",
            F.to_date(F.col("follow_up_target_date").cast("string")),
        )
        .withColumn("_days_to_target", F.datediff("_target_date", "_intake_date"))
    )
    follow_up_df.agg(
        F.count("_days_to_target").alias("records_with_both_dates"),
        F.avg("_days_to_target").alias("mean_days_to_target"),
        F.expr("percentile_approx(_days_to_target, 0.5)").alias("median_days_to_target"),
        F.sum((F.col("_days_to_target") < 0).cast("long")).alias("negative_intervals"),
    ).show(truncate=False)
else:
    print("One or both follow-up date fields were not found.")

## 10. Compare selected categories

A grouped count can help identify patterns worth investigating. The minimum group size filter reduces the chance of exposing rare cases; do not share small-cell results without instructor approval.

In [ ]:
if "program_type" in eda_df.columns and "referral_status" in eda_df.columns:
    (
        eda_df.groupBy("program_type", "referral_status")
        .count()
        .filter(F.col("count") >= 5)
        .orderBy(F.desc("count"))
        .show(n=50, truncate=True)
    )
else:
    print("program_type or referral_status was not found.")

## 11. Questions to take forward

After running the notebook, use your results to decide which questions merit deeper analysis. Examples:

- Are intake volumes changing over time, and are there gaps in date coverage?
- Which categories have the most missing data?
- How do follow-up target intervals vary, and how many records have a negative interval?
- Do the category patterns persist after checking for missing values and small group sizes?

Document limitations (inferred types, date parsing, missingness, and any suppressed categories). Keep raw records, names, notes, and encoded photos out of shared notebook outputs.